# Merge script

Assembles the five source notebooks into one submission file. This notebook is a build
tool. It is not part of the submission and is not merged into the output.

It reads only. Nothing is retrained, no result is recomputed, and no source notebook is
modified.

**Do not run the merged output.** Executing it top to bottom would retrain all thirty runs
and discard the point of splitting the work.

In [1]:
GROUP_NO = "02"
STUDENT_IDS = ["23201151", "23201409", "23201042", "23201136"]

SOURCES = [
    "01_phase1_mahir.ipynb",      # sections 1 to 7
    "03_rnn_hasib.ipynb",         # sections 8, 9.1, 9.2
    "04_lstm_fabiha.ipynb",       # sections 9.3, 9.4
    "05_gru_bert_zawad.ipynb",    # sections 9.5, 9.6, 10
    "06_final_analysis.ipynb",    # sections 11 to 15
]

OUT_NAME = f"{GROUP_NO}_{'_'.join(STUDENT_IDS)}.ipynb"
print(f"output file: {OUT_NAME}")
print(f"sources    : {len(SOURCES)}")

output file: 02_23201151_23201409_23201042_23201136.ipynb
sources    : 5


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import nbformat
from pathlib import Path
from uuid import uuid4

DATA_DIR = Path("/content/drive/MyDrive/cse440_project")
START, END = "=== CONTRIB START ===", "=== CONTRIB END ==="

missing = [f for f in SOURCES if not (DATA_DIR / f).exists()]
assert not missing, f"not found in Drive: {missing}"

for f in SOURCES:
    nb = nbformat.read(DATA_DIR / f, as_version=4)
    has_start = any(START in c.source for c in nb.cells)
    has_end   = any(END in c.source for c in nb.cells)
    code_n    = sum(1 for c in nb.cells if c.cell_type == "code")
    no_out    = sum(1 for c in nb.cells
                    if c.cell_type == "code" and not c.get("outputs"))
    print(f"{f:28s} cells {len(nb.cells):3d}  code {code_n:3d}  "
          f"no-output {no_out:2d}  START {has_start}  END {has_end}")
    assert has_start and has_end, f"{f} is missing a sentinel"

Mounted at /content/drive
01_phase1_mahir.ipynb        cells  58  code  32  no-output  1  START True  END True
03_rnn_hasib.ipynb           cells  39  code  25  no-output  0  START True  END True
04_lstm_fabiha.ipynb         cells  50  code  34  no-output  2  START True  END True
05_gru_bert_zawad.ipynb      cells  62  code  32  no-output  0  START True  END True
06_final_analysis.ipynb      cells  33  code  14  no-output  0  START True  END True


## Extract the contributed regions

Everything between the two sentinels is contributed content. Everything outside is a
local copy of the shared setup, which the Phase 0 block already provides once.

Both sentinel cells are dropped. The `continue` after each match is what keeps them out
of the output.

In [3]:
def contributed(path):
    nb = nbformat.read(path, as_version=4)
    cells, inside = [], False
    for c in nb.cells:
        if START in c.source:
            inside = True
            continue
        if END in c.source:
            inside = False
            continue
        if inside:
            cells.append(c)
    if not cells:
        raise ValueError(f"no cells found between sentinels in {path.name}")
    return cells


blocks = {}
for f in SOURCES:
    cells = contributed(DATA_DIR / f)
    blocks[f] = cells
    code_n = sum(1 for c in cells if c.cell_type == "code")
    print(f"{f:28s} contributes {len(cells):3d} cells ({code_n:3d} code)")

total = sum(len(v) for v in blocks.values())
print(f"\ntotal contributed cells: {total}")

01_phase1_mahir.ipynb        contributes  56 cells ( 32 code)
03_rnn_hasib.ipynb           contributes  32 cells ( 20 code)
04_lstm_fabiha.ipynb         contributes  39 cells ( 25 code)
05_gru_bert_zawad.ipynb      contributes  47 cells ( 26 code)
06_final_analysis.ipynb      contributes  31 cells ( 14 code)

total contributed cells: 205


## Assemble

Cell ids are regenerated because the source notebooks were saved under different nbformat
minor versions. Mixing cells with and without an `id` field produces a file that fails
validation.

Execution counts are renumbered in a single sequence. Each notebook restarted its own
counter at 1, so without this the merged file shows four separate sequences. This
relabels only. No cell is executed and every output is preserved.

In [4]:
master = nbformat.v4.new_notebook()
master.metadata = {
    "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
    "language_info": {"name": "python"},
    "colab": {"provenance": [], "toc_visible": True},
}

for f in SOURCES:
    master.cells.extend(blocks[f])

for c in master.cells:
    c["id"] = uuid4().hex[:8]

n = 1
for c in master.cells:
    if c.cell_type == "code":
        c.execution_count = n
        for o in c.get("outputs", []):
            if "execution_count" in o:
                o["execution_count"] = n
        n += 1

print(f"assembled {len(master.cells)} cells, {n - 1} code cells renumbered")

assembled 205 cells, 117 code cells renumbered


## Verify before writing

Four checks. Every code cell carries output, since the brief states that cells without
output cannot be verified. No cell holds an error traceback. The section headings appear
once each and in order. And the sentinels are gone.

In [5]:
code_cells = [c for c in master.cells if c.cell_type == "code"]
no_output  = [i for i, c in enumerate(master.cells)
              if c.cell_type == "code" and not c.get("outputs")]
errors     = [i for i, c in enumerate(master.cells)
              for o in c.get("outputs", [])
              if o.get("output_type") == "error"]
leftover   = [i for i, c in enumerate(master.cells)
              if START in c.source or END in c.source]

print(f"total cells      : {len(master.cells)}")
print(f"code cells       : {len(code_cells)}")
print(f"markdown cells   : {len(master.cells) - len(code_cells)}")
print(f"without output   : {len(no_output)} {no_output if no_output else ''}")
print(f"error outputs    : {len(errors)} {errors if errors else ''}")
print(f"sentinels left   : {len(leftover)}")

assert not errors, "the merged notebook contains error output"
assert not leftover, "a sentinel survived the merge"
if no_output:
    print("\nnote: cells listed above are function definitions with nothing to print")

total cells      : 205
code cells       : 117
markdown cells   : 88
without output   : 3 [2, 88, 89]
error outputs    : 0 
sentinels left   : 0

note: cells listed above are function definitions with nothing to print


In [6]:
print("SECTION HEADINGS IN ORDER")
print("=" * 74)
for c in master.cells:
    if c.cell_type != "markdown":
        continue
    for line in c.source.split("\n"):
        s = line.strip()
        if s.startswith("#"):
            depth = len(s) - len(s.lstrip("#"))
            print(f"{'  ' * (depth - 1)}{s.lstrip('# ')[:70]}")
            break

SECTION HEADINGS IN ORDER
Multi-Class News Article Classification Using Machine Learning and Nat
  Section 2 - Setup and Imports
  Section 3 - Dataset Loading and Description
    Hierarchical label structure
  Section 4 - Exploratory Data Analysis
    4.1 Class distribution
    4.2 Document length
    4.3 Data quality
    4.4 Most frequent terms per family
  Section 5 - Preprocessing
    5.1 Comparing the two strategies
  Section 6 - Train / Validation / Test Split
  Section 7 - Text Representation
    7.1 TF-IDF
    7.2 Keras tokenizer, the index both embedding matrices align to
    7.3 Word2Vec, trained on this corpus
    7.4 GloVe, pretrained on general text
    7.5 Exported artifacts
    Verification
  Section 8 - Machine Learning Models
    8.4 Preprocessing ablation
  Sections 9.1 and 9.2 - SimpleRNN and Bidirectional SimpleRNN
    Learning curves
    The embedding contrast, isolated
  Validation leaderboard
  Test evaluation
    Classification reports
    Confusion matrices
    

## Write the submission file

Written to the Drive root next to the sources. Download it from there for the Google Form.

In [7]:
out_path = DATA_DIR / OUT_NAME
nbformat.write(master, out_path)
nbformat.validate(nbformat.read(out_path, as_version=4))

size_mb = out_path.stat().st_size / 1e6
print(f"written : {out_path}")
print(f"size    : {size_mb:.1f} MB")
print(f"cells   : {len(master.cells)}")
print("nbformat validation: PASS")

if size_mb > 45:
    print("\nwarning: large file, mostly embedded figures. "
          "It will still upload but may be slow to open.")

written : /content/drive/MyDrive/cse440_project/02_23201151_23201409_23201042_23201136.ipynb
size    : 3.8 MB
cells   : 205
nbformat validation: PASS
